# L9：神经网络基础


# L9：神经网络基础

## 学习目标

1. 理解感知机（Perceptron）模型的结构与工作原理
2. 掌握反向传播（Backpropagation）算法的数学推导与实现
3. 熟悉常用激活函数的特性、公式与适用场景
4. 理解损失函数的设计原理与选择策略
5. 能够从零实现一个完整的多层感知机

## 核心知识点

### 1. 感知机（Perceptron）

感知机是最基本的神经网络单元，由 Rosenblatt 于1957年提出。其核心结构如下：


In [ ]:
输入向量 x = [x₁, x₂, ..., xₙ]
权重向量 w = [w₁, w₂, ..., wₙ]
偏置 b
输出 y = f(w·x + b) = f(Σwᵢxᵢ + b)


其中 f 为激活函数。感知机本质上是一个线性分类器，通过调整权重与偏置来学习分类边界。

**感知机的局限**：单层感知机无法解决异或（XOR）等非线性可分问题，这直接推动了多层神经网络的发展。

### 2. 多层感知机（MLP）结构


In [ ]:
输入层 → 隐藏层1 → 隐藏层2 → ... → 隐藏层L → 输出层


每层的输出作为下一层的输入：
- 第 l 层的激活值：a^(l) = f(z^(l))，其中 z^(l) = W^(l)·a^(l-1) + b^(l)
- W^(l) 为第 l 层的权重矩阵，b^(l) 为偏置向量

### 3. 激活函数

#### Sigmoid 函数

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

特点：输出范围(0,1)，可表示概率，但容易出现梯度消失问题。

#### Tanh 函数

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

特点：输出范围(-1,1)，零中心化，收敛比Sigmoid快，但仍存在梯度消失问题。

#### ReLU 函数（Rectified Linear Unit）

$$f(z) = \max(0, z)$$

特点：计算高效，缓解梯度消失问题，是目前最广泛使用的激活函数。变体包括 Leaky ReLU、PReLU、ELU 等。

#### Softmax 函数（用于多分类输出层）

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$

特点：将任意实数向量转换为概率分布，所有输出和为1。

### 4. 前向传播（Forward Propagation）

给定输入样本 x，经过各层网络的计算：


In [ ]:
z^(1) = W^(1)x + b^(1)
a^(1) = f(z^(1))
z^(2) = W^(2)a^(1) + b^(2)
a^(2) = f(z^(2))
...
y = a^(L)


### 5. 损失函数

#### 均方误差（MSE）

$$L_{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

适用于回归问题。

#### 交叉熵损失（Cross-Entropy）

$$L_{CE} = -\sum_{i} y_i \log(\hat{y}_i)$$

适用于分类问题。相比MSE，收敛更快，梯度更稳定（不会因输出接近1而梯度趋于0）。

#### 二元交叉熵（Binary Cross-Entropy）

$$L_{BCE} = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

适用于二分类问题。

### 6. 反向传播（Backpropagation）

反向传播是训练神经网络的核心理论基础。其核心思想是利用链式法则，从输出层向输入层逐层计算梯度。

#### 链式法则回顾

若 y = f(g(x))，则：
$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

#### 输出层梯度

对于交叉熵损失 + Softmax（常见组合）：

假设 z 是 softmax 的输入，y 是真实标签，$\hat{y}$ 是预测概率。

$$\frac{\partial L}{\partial z_i} = \hat{y}_i - y_i$$

这个简洁的结果是softmax+交叉熵组合的特有性质。

#### 隐藏层梯度

对于第 l 层：

$$\frac{\partial L}{\partial W^{(l)}} = \frac{\partial L}{\partial z^{(l)}} \cdot \frac{\partial z^{(l)}}{\partial W^{(l)}} = \delta^{(l)} \cdot a^{(l-1)T}$$

其中 $\delta^{(l)} = \frac{\partial L}{\partial z^{(l)}}$ 称为"敏感性"（sensitivity）。

反向传播的递推公式：

$$\delta^{(l)} = (W^{(l+1)T} \cdot \delta^{(l+1)}) \odot f'(z^{(l)})$$

其中 $\odot$ 表示逐元素乘法（Hadamard product）。

### 7. 梯度下降优化

参数更新规则：

$$W^{(l)} := W^{(l)} - \alpha \frac{\partial L}{\partial W^{(l)}}$$

其中 $\alpha$ 为学习率（learning rate）。

常用优化器包括：SGD、Momentum、RMSProp、Adam 等。

## 代码示例

### 完整的多层感知机实现


In [ ]:
import numpy as np

# ============================================================
# 激活函数及其导数
# ============================================================

def sigmoid(z):
    """Sigmoid激活函数"""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_grad(z):
    """Sigmoid的梯度（关于激活值a，而非原始z）"""
    return z * (1.0 - z)

def relu(z):
    """ReLU激活函数"""
    return np.maximum(0, z)

def relu_grad(z):
    """ReLU的梯度"""
    return (z > 0).astype(float)

def softmax(z):
    """Softmax激活函数（数值稳定版）"""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# ============================================================
# 神经网络类
# ============================================================

class NeuralNetwork:
    """
    多层感知机（Multi-Layer Perceptron）
    支持任意层数和任意神经元数量的网络结构
    """

    def __init__(self, layer_dims, activation='relu', output_activation='softmax'):
        """
        参数：
            layer_dims: 列表，如 [784, 256, 128, 10] 表示输入784维，隐藏层256/128，输出10维
            activation: 隐藏层激活函数，可选 'relu' 或 'sigmoid'
            output_activation: 输出层激活函数，可选 'softmax' 或 'sigmoid'
        """
        self.layer_dims = layer_dims
        self.num_layers = len(layer_dims)
        self.activation = activation
        self.output_activation = output_activation

        # Xavier初始化权重
        self.weights = {}
        self.biases = {}
        for l in range(1, self.num_layers):
            # Xavier初始化公式：sqrt(2.0 / fan_in)
            scale = np.sqrt(2.0 / layer_dims[l-1])
            self.weights[l] = np.random.randn(layer_dims[l-1], layer_dims[l]) * scale
            self.biases[l] = np.zeros((1, layer_dims[l]))

    def _forward_layer(self, z, activation):
        """单层前向传播"""
        if activation == 'relu':
            return relu(z), relu_grad
        elif activation == 'sigmoid':
            return sigmoid(z), sigmoid_grad
        elif activation == 'linear':
            return z, lambda x: np.ones_like(x)
        else:
            raise ValueError(f"Unknown activation: {activation}")

    def forward(self, X):
        """
        前向传播
        返回：
            caches: 各层的中间结果，用于反向传播
        """
        caches = {}
        A = X
        caches[0] = {'A': A}

        # 隐藏层
        for l in range(1, self.num_layers - 1):
            Z = A @ self.weights[l] + self.biases[l]
            A, grad_fn = self._forward_layer(Z, self.activation)
            caches[l] = {'Z': Z, 'A': A, 'grad_fn': grad_fn}

        # 输出层
        l = self.num_layers - 1
        Z = A @ self.weights[l] + self.biases[l]
        if self.output_activation == 'softmax':
            A = softmax(Z)
        elif self.output_activation == 'sigmoid':
            A = sigmoid(Z)
        caches[l] = {'Z': Z, 'A': A}

        return caches

    def compute_loss(self, Y_pred, Y_true):
        """计算交叉熵损失（数值稳定版）"""
        m = Y_true.shape[0]
        # 将标签转换为one-hot编码（如果还不是）
        if Y_true.ndim == 1 or Y_true.shape[1] == 1:
            Y_onehot = np.zeros((m, Y_pred.shape[1]))
            Y_onehot[np.arange(m), Y_true.astype(int).flatten()] = 1
            Y_true = Y_onehot

        # Clip防止log(0)
        Y_pred = np.clip(Y_pred, 1e-10, 1 - 1e-10)
        loss = -np.sum(Y_true * np.log(Y_pred)) / m
        return loss

    def backward(self, Y_true, caches):
        """
        反向传播
        参数：
            Y_true: 真实标签（one-hot编码）
            caches: 前向传播的中间结果
        返回：
            grads: 各层梯度字典
        """
        m = Y_true.shape[0]
        grads = {}

        # 输出层梯度（交叉熵 + Softmax 的特殊形式）
        L = self.num_layers - 1
        A_L = caches[L]['A']
        delta = A_L - Y_true  # shape: (m, num_classes)

        grads[L] = {
            'dW': caches[L-1]['A'].T @ delta / m,
            'db': np.sum(delta, axis=0, keepdims=True) / m
        }

        # 隐藏层反向传播
        for l in range(L - 1, 0, -1):
            delta = (delta @ self.weights[l+1].T) * caches[l]['grad_fn'](caches[l]['A'])
            grads[l] = {
                'dW': caches[l-1]['A'].T @ delta / m,
                'db': np.sum(delta, axis=0, keepdims=True) / m
            }

        return grads

    def update_params(self, grads, learning_rate):
        """使用梯度下降更新参数"""
        for l in range(1, self.num_layers):
            self.weights[l] -= learning_rate * grads[l]['dW']
            self.biases[l] -= learning_rate * grads[l]['db']

    def fit(self, X, Y, epochs=100, learning_rate=0.01, batch_size=32, verbose=True):
        """
        训练神经网络
        参数：
            X: 训练数据 (m, n_features)
            Y: 标签 (m,) 或 (m, n_classes) 的one-hot编码
            epochs: 训练轮数
            learning_rate: 学习率
            batch_size: 小批量大小
            verbose: 是否打印训练进度
        """
        m = X.shape[0]

        # 转为one-hot编码
        if Y.ndim == 1:
            Y_onehot = np.zeros((m, self.layer_dims[-1]))
            Y_onehot[np.arange(m), Y.astype(int).flatten()] = 1
            Y = Y_onehot

        history = {'loss': []}

        for epoch in range(epochs):
            # 打乱数据
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            Y_shuffled = Y[indices]

            # Mini-batch训练
            epoch_loss = 0
            num_batches = 0
            for start in range(0, m, batch_size):
                end = min(start + batch_size, m)
                X_batch = X_shuffled[start:end]
                Y_batch = Y_shuffled[start:end]

                # 前向传播
                caches = self.forward(X_batch)
                Y_pred = caches[self.num_layers - 1]['A']

                # 计算损失
                loss = self.compute_loss(Y_pred, Y_batch)
                epoch_loss += loss
                num_batches += 1

                # 反向传播
                grads = self.backward(Y_batch, caches)

                # 更新参数
                self.update_params(grads, learning_rate)

            avg_loss = epoch_loss / num_batches
            history['loss'].append(avg_loss)

            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

        return history

    def predict(self, X):
        """预测类别"""
        caches = self.forward(X)
        return np.argmax(caches[self.num_layers - 1]['A'], axis=1)

    def accuracy(self, X, Y):
        """计算准确率"""
        predictions = self.predict(X)
        if Y.ndim > 1:
            Y = np.argmax(Y, axis=1)
        return np.mean(predictions == Y.astype(int).flatten())


# ============================================================
# 示例：MNIST手写数字识别
# ============================================================

if __name__ == "__main__":
    # 模拟数据加载（实际使用时应从sklearn或torchvision加载真实数据）
    from sklearn.datasets import fetch_openml
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    print("加载MNIST数据集...")
    mnist = fetch_openml('mnist_784', version=1, as_frame=False)
    X, y = mnist.data, mnist.target.astype(int)

    # 归一化到[0,1]
    X = X / 255.0

    # 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    print(f"训练集大小: {X_train.shape[0]}, 测试集大小: {X_test.shape[0]}")
    print(f"特征维度: {X_train.shape[1]}, 类别数: {len(np.unique(y))}")

    # 创建网络：输入784 -> 隐藏层256 -> 隐藏层128 -> 输出10
    nn = NeuralNetwork([784, 256, 128, 10], activation='relu', output_activation='softmax')

    # 训练
    print("\n开始训练...")
    history = nn.fit(X_train, y_train, epochs=50, learning_rate=0.1, batch_size=64, verbose=True)

    # 评估
    train_acc = nn.accuracy(X_train, y_train)
    test_acc = nn.accuracy(X_test, y_test)
    print(f"\n训练集准确率: {train_acc:.4f}")
    print(f"测试集准确率: {test_acc:.4f}")


### 反向传播算法详细推导实现


In [ ]:
"""
展示反向传播算法的完整数学推导过程
以一个简单的3层网络为例：输入2维 -> 隐藏层4维 -> 输出2维
"""

import numpy as np

class BackpropagationDemo:
    """
    手动实现反向传播的每一步计算，展示链式法则的具体应用
    """

    def __init__(self):
        np.random.seed(42)
        # 初始化权重（故意选择较小的值以便手动验证）
        self.W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
        self.b1 = np.array([[0.1, 0.2, 0.3, 0.4]])
        self.W2 = np.array([[0.1, 0.2, 0.3, 0.4],
                           [0.5, 0.6, 0.7, 0.8]])
        self.b2 = np.array([[0.1, 0.2]])

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))

    def sigmoid_derivative(self, z):
        """sigmoid导数（关于原始输入z）"""
        s = self.sigmoid(z)
        return s * (1 - s)

    def forward(self, X):
        """前向传播"""
        # 第一层
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.sigmoid(self.z1)
        # 第二层（输出层）
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def compute_loss(self, y_pred, y_true):
        """MSE损失"""
        return 0.5 * np.sum((y_pred - y_true) ** 2)

    def backward(self, X, y_true):
        """
        反向传播
        展示链式法则的每一步计算
        """
        m = X.shape[0]  # 样本数

        # ========== 输出层梯度 ==========
        # L = 0.5 * (y - y_pred)^2
        # dL/dy_pred = y_pred - y_true
        dL_dypred = self.a2 - y_true

        # dy_pred/dz2 = sigmoid'(z2)
        dy_pred_dz2 = self.sigmoid_derivative(self.z2)

        # delta2 = dL/dz2 = dL/dy_pred * dy_pred/dz2
        delta2 = dL_dypred * dy_pred_dz2

        # dL/dW2 = a1.T @ delta2
        dL_dW2 = self.a1.T @ delta2 / m

        # dL/db2 = sum(delta2) / m
        dL_db2 = np.sum(delta2, axis=0, keepdims=True) / m

        # ========== 隐藏层梯度 ==========
        # dL/da1 = delta2 @ W2.T
        dL_da1 = delta2 @ self.W2.T

        # da1/dz1 = sigmoid'(z1)
        da1_dz1 = self.sigmoid_derivative(self.z1)

        # delta1 = dL/dz1 = dL/da1 * da1/dz1
        delta1 = dL_da1 * da1_dz1

        # dL/dW1 = X.T @ delta1
        dL_dW1 = X.T @ delta1 / m

        # dL/db1 = sum(delta1) / m
        dL_db1 = np.sum(delta1, axis=0, keepdims=True) / m

        return {
            'dW1': dL_dW1, 'db1': dL_db1,
            'dW2': dL_dW2, 'db2': dL_db2
        }

    def fit(self, X, y, epochs=1000, lr=1.0):
        """训练循环"""
        for epoch in range(epochs):
            # 前向传播
            y_pred = self.forward(X)
            loss = self.compute_loss(y_pred, y)

            # 反向传播
            grads = self.backward(X, y)

            # 更新权重
            self.W1 -= lr * grads['dW1']
            self.b1 -= lr * grads['db1']
            self.W2 -= lr * grads['dW2']
            self.b2 -= lr * grads['db2']

            if (epoch + 1) % 200 == 0:
                print(f"Epoch {epoch+1}, Loss: {loss:.6f}")

    def predict(self, X):
        return self.forward(X)


# 验证示例
if __name__ == "__main__":
    demo = BackpropagationDemo()

    # 简单的训练数据：X=[[0.5, 0.1], [0.2, 0.3]], y=[[0.7, 0.3], [0.2, 0.8]]
    X = np.array([[0.5, 0.1], [0.2, 0.3]])
    y = np.array([[0.7, 0.3], [0.2, 0.8]])

    print("训练前预测:")
    print(demo.forward(X))

    demo.fit(X, y, epochs=1000, lr=3.0)

    print("\n训练后预测:")
    print(demo.forward(X))

    print("\n真实标签:")
    print(y)


## 练习题

1. **感知机理解**：假设有一个二维输入的感知机，权重 w=[0.5,-0.3]，偏置 b=-0.1。请计算输入 x=[2,1] 时的输出（假设使用阶跃函数作为激活函数）。

2. **梯度计算**：对于一个使用MSE损失的单层网络，假设有两个样本的预测值分别为[0.9, 0.1]和[0.2, 0.8]，对应的真实标签为[1,0]和[0,1]。请手工计算损失值和输出层的梯度。

3. **激活函数选择**：某同学设计了一个5层神经网络用于图像分类，但训练时发现梯度几乎为0，收敛极慢。请分析可能的原因，并提出至少两种解决方案。

4. **反向传播推导**：考虑一个三层网络：输入->隐藏层(tanh)->输出层(softmax+CE)。请写出反向传播时输出层到隐藏层的梯度计算公式（不需要写出具体数值）。

5. **MLP实现**：请设计一个MLP来实现异或（XOR）函数。要求网络至少有一个隐藏层，激活函数使用ReLU。请给出网络结构（各层神经元数量）和对应的权重初始化范围。

## 延伸阅读

- Rosenblatt, F. (1958). "The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain". Psychological Review.
- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). "Learning representations by back-propagating errors". Nature.
- LeCun, Y., Bottou, L., Orr, G. B., & Muller, K. R. (2012). "Efficient BackProp" in Neural Networks: Tricks of the Trade, Springer.
- 深度学习圣经：Ian Goodfellow, Yoshua Bengio, Aaron Courville. "Deep Learning" Chapter 6.

---
